In [0]:
import os, json, time, requests

base = os.getcwd()   # notebook ka folder (Git folder)
print(base)

core_indicators = {
    "NY.GDP.MKTP.CD":       "GDP (current US$)",
    "NY.GDP.PCAP.CD":       "GDP per capita (current US$)",
    "NY.GDP.MKTP.KD.ZG":    "GDP growth (annual %)",
    "SP.POP.TOTL":          "Population, total",
    "SP.DYN.LE00.IN":       "Life expectancy at birth, total (years)",
    "SE.ADT.LITR.ZS":       "Literacy rate, adult total (% of people ages 15 and above)",
    "SH.DYN.MORT":          "Mortality rate, under-5 (per 1,000 live births)",
    "SH.XPD.CHEX.GD.ZS":    "Current health expenditure (% of GDP)",
    "SE.PRM.ENRR":          "School enrollment, primary (% gross)",
    "SE.SEC.ENRR":          "School enrollment, secondary (% gross)",
    "IT.NET.USER.ZS":       "Individuals using the Internet (% of population)",
    "EN.ATM.CO2E.PC":       "CO2 emissions (metric tons per capita)",
    "EN.GHG.CO2.PC.CE.AR5": "CO2 emissions excluding LULUCF per capita",
    "SI.POV.DDAY":          "Poverty headcount ratio",
    "FP.CPI.TOTL.ZG":       "Inflation, consumer prices (annual %)",
    "SL.UEM.TOTL.ZS":       "Unemployment, total (% of labor force)",
}

def pull_indicator(code, start=1960, end=2024, retries=3):
    url = f"https://api.worldbank.org/v2/country/all/indicator/{code}"
    params = {"format": "json", "date": f"{start}:{end}", "per_page": 20000, "page": 1}
    records = []
    while True:
        data = None
        for attempt in range(retries):
            try:
                r = requests.get(url, params=params, timeout=60)
                r.raise_for_status()
                data = r.json()
                break
            except Exception as e:
                if attempt == retries - 1:
                    print(f"  ! {code}: {e}")
                    return records
                time.sleep(2 * (attempt + 1))
        if not isinstance(data, list) or len(data) < 2 or not data[1]:
            return records            # galat code ya koi data nahi
        records.extend(data[1])
        if params["page"] >= int(data[0].get("pages", 1)):
            return records
        params["page"] += 1

full_core, summary = [], []
for code, name in core_indicators.items():
    recs = pull_indicator(code)
    non_null = sum(1 for r in recs if r.get("value") is not None)
    summary.append((code, name, len(recs), non_null))
    print(f"{code:22s} records={len(recs):6d}  non-null={non_null:6d}  {name}")
    if non_null > 0:                 # jis indicator mein koi value nahi, wo chhod do
        full_core.extend(recs)
    time.sleep(0.3)

# Incremental = wahi data ke sirf 2023-2024 (isliye values full load se bilkul same hongi)
incremental_core = [r for r in full_core if r["date"] in ("2023", "2024")]

with open(f"{base}/full_load_core.json", "w") as f:
    json.dump(full_core, f)
with open(f"{base}/incremental_core.json", "w") as f:
    json.dump(incremental_core, f)
with open(f"{base}/indicators_core.json", "w") as f:
    json.dump({c: n for c, n, _, nn in summary if nn > 0}, f, indent=2)

for fn in ["full_load_core.json", "incremental_core.json"]:
    print(fn, round(os.path.getsize(f"{base}/{fn}") / 1024 / 1024, 2), "MB")
print("full records:", len(full_core), "| incremental records:", len(incremental_core))